In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('SeoulBikeData.csv', encoding='unicode_escape')
df.columns = ['Date', 'Rented Bike Count', 'Hour', 'Temperature', 'Humidity', 
              'Wind_speed', 'Visibility', 'Dew_point_temp', 'Solar_Radiation', 
              'Rainfall', 'Snowfall', 'Seasons', 'Holiday', 'Functioning Day']

In [ ]:
# 조건부확률 1: 날씨가 좋으면 얼마나 많이 빌릴까

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
weekday_df = df[df['Date'].dt.dayofweek < 5]
weekend_df = df[df['Date'].dt.dayofweek >= 5]

# 2. 조건부 확률 계산 함수 정의
def get_weather_prob(data, label):
    # 사건 A: 높은 수요 (1000대 이상)
    cond_A = data['Rented Bike Count'] >= 1000
    
    # 조건 W: 완벽한 날씨 (기온 15~25도, 비/눈 없음)
    cond_W = (data['Temperature'] >= 15) & (data['Temperature'] <= 25) & \
             (data['Rainfall'] == 0) & (data['Snowfall'] == 0)
    
    # P(A): 해당 그룹의 평소 높은 수요 확률
    p_A = cond_A.mean()
    
    # P(A|W): 날씨가 완벽할 때(W) 높은 수요(A)가 발생할 확률
    perfect_weather_days = data[cond_W]
    p_A_given_W = (perfect_weather_days['Rented Bike Count'] >= 1000).mean()
    
    # 결과 출력
    print(f"[{label} 분석 결과]")
    print(f"평소 높은 수요 확률 P(A)     : {p_A:.4f} ({p_A*100:.1f}%)")
    print(f"날씨 좋을 때 수요 확률 P(A|W): {p_A_given_W:.4f} ({p_A_given_W*100:.1f}%)")
    print(f"확률 증가폭                : {p_A_given_W - p_A:.4f} (+{(p_A_given_W - p_A)*100:.1f}%p)\n")

# 3. 함수 실행
get_weather_prob(weekday_df, "평일(Weekday)")
get_weather_prob(weekend_df, "주말(Weekend)")

[평일(Weekday) 분석 결과]
평소 높은 수요 확률 P(A)     : 0.2840 (28.4%)
날씨 좋을 때 수요 확률 P(A|W): 0.5064 (50.6%)
확률 증가폭                : 0.2224 (+22.2%p)

[주말(Weekend) 분석 결과]
평소 높은 수요 확률 P(A)     : 0.2568 (25.7%)
날씨 좋을 때 수요 확률 P(A|W): 0.4985 (49.9%)
확률 증가폭                : 0.2417 (+24.2%p)



In [ ]:
# 2. 비가 오면 이용량이 얼마나 줄어들까?
active_df = df[df['Functioning Day'] == 'Yes'].copy()
overall_avg = active_df['Rented Bike Count'].mean()

def analyze_rain_penalty(threshold):
    # 조건: 강수량이 threshold 이상인 상황
    condition_mask = active_df['Rainfall'] >= threshold
    subset = active_df[condition_mask]
    
    if len(subset) == 0:
        return
        
    # P(Condition): 해당 강수량 조건이 발생할 확률
    p_cond = len(subset) / len(active_df)
    
    # 사건 A: 수요 반토막 (대여량이 전체 평균의 50% 미만)
    # P(A|Rain): 해당 비가 올 때 수요가 반토막 날 확률
    p_half_demand = (subset['Rented Bike Count'] < overall_avg * 0.5).mean()
    avg_rent = subset['Rented Bike Count'].mean()
    
    print(f"--- 조건: 강수량이 {threshold}mm 이상일 때 ---")
    print(f"1) 발생 확률: {p_cond*100:.2f}%")
    print(f"2) 수요가 평소의 절반 이하로 '폭락'할 확률: {p_half_demand*100:.1f}%")
    print(f"3) 이때의 평균 대여량: {avg_rent:,.1f}대 (평소의 {(avg_rent/overall_avg)*100:.1f}% 수준)\n")

print(f"전체 운영 시간 평균 대여량: {overall_avg:,.1f}대\n")
for t in [0.1, 5, 10]:
    analyze_rain_penalty(t)

전체 운영 시간 평균 대여량: 729.2대

--- 조건: 강수량이 0.1mm 이상일 때 ---
1) 발생 확률: 6.10%
2) 수요가 평소의 절반 이하로 '폭락'할 확률: 87.8%
3) 이때의 평균 대여량: 167.3대 (평소의 22.9% 수준)

--- 조건: 강수량이 5mm 이상일 때 ---
1) 발생 확률: 0.82%
2) 수요가 평소의 절반 이하로 '폭락'할 확률: 95.7%
3) 이때의 평균 대여량: 74.6대 (평소의 10.2% 수준)

--- 조건: 강수량이 10mm 이상일 때 ---
1) 발생 확률: 0.26%
2) 수요가 평소의 절반 이하로 '폭락'할 확률: 95.5%
3) 이때의 평균 대여량: 86.5대 (평소의 11.9% 수준)



In [ ]:
# 3. 평일 출퇴근 시간의 고수요 집중도 분석

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
weekday_only = df[(df['Date'].dt.dayofweek < 5) & (df['Holiday'] == 'No Holiday')].copy()

def analyze_rush_hour_peak(data):
    # 사건 A: 고수요 발생 (대여량 1,500대 이상)
    condition_A = data['Rented Bike Count'] >= 1500
    
    # 조건 B: 출퇴근 시간대 (오전 8시, 오후 6시)
    condition_B = data['Hour'].isin([8, 18])
    
    # 조건부 확률 계산
    # P(A): 평일 전체 시간 중 고수요가 발생할 확률
    p_A = condition_A.mean()
    
    # P(A|B): 평일 출퇴근 시간 중 고수요가 발생할 확률
    subset_B = data[condition_B]
    p_A_given_B = (subset_B['Rented Bike Count'] >= 1500).mean()
    
    # 평균 대여량 비교 (부가 정보)
    avg_weekday = data['Rented Bike Count'].mean()
    avg_rush_hour = subset_B['Rented Bike Count'].mean()

    print(f"====== 평일 출퇴근 고수요 집중 분석 ======")
    print(f"[1. 조건부 확률 분석]")
    print(f"- 평일 일반적인 고수요 확률 P(A)     : {p_A:.4f} ({p_A*100:.1f}%)")
    print(f"- 출퇴근 시간대 고수요 확률 P(A|B) : {p_A_given_B:.4f} ({p_A_given_B*100:.1f}%)")
    print(f"- 고수요 발생 위험도 증가폭        : {p_A_given_B / p_A:.2f}배")
    
    print(f"\n[2. 대여량 비교]")
    print(f"- 평일 전체 평균 대여량 : {avg_weekday:,.1f}대")
    print(f"- 출퇴근 시간 평균 대여량 : {avg_rush_hour:,.1f}대")
    print(f"- 평소 대비 대여량 증가율 : {((avg_rush_hour/avg_weekday)-1)*100:.1f}%")
    print(f"===========================================")

# 2. 실행
analyze_rush_hour_peak(weekday_only)

====== 평일 출퇴근 고수요 집중 분석 ======
[1. 조건부 확률 분석]
- 평일 일반적인 고수요 확률 P(A)     : 0.1334 (13.3%)
- 출퇴근 시간대 고수요 확률 P(A|B) : 0.5506 (55.1%)
- 고수요 발생 위험도 증가폭        : 4.13배

[2. 대여량 비교]
- 평일 전체 평균 대여량 : 728.6대
- 출퇴근 시간 평균 대여량 : 1,503.1대
- 평소 대비 대여량 증가율 : 106.3%


In [ ]:
# 4. '휴일'의 정체성: 평일 공휴일은 평일인가 주말인가?

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

def analyze_holiday_identity(data):
    # 그룹 A: 일반 평일 (월~금 & 휴일 아님) -> 기준점(Baseline)
    regular_weekday = data[(data['Date'].dt.dayofweek < 5) & (data['Holiday'] == 'No Holiday')]
    
    # 그룹 B: 일반 주말 (토, 일) -> 레저 패턴의 대표군
    regular_weekend = data[data['Date'].dt.dayofweek >= 5]
    
    # 그룹 C: 평일 공휴일 (월~금 인데 휴일임) -> 우리가 정체를 밝혀야 할 타겟군
    weekday_holiday = data[(data['Date'].dt.dayofweek < 5) & (data['Holiday'] == 'Holiday')]
    
    # 조건부 확률 계산 함수
    def get_demand_probs(subset):
        # 사건 1: 출퇴근 고수요 방어 (8시, 18시에 1500대 이상 빌릴 확률)
        commute_cond = subset['Hour'].isin([8, 18])
        p_commute = (subset[commute_cond]['Rented Bike Count'] >= 1500).mean()
        
        # 사건 2: 오후 레저 고수요 (14시~17시에 1000대 이상 빌릴 확률)
        leisure_cond = subset['Hour'].isin([14, 15, 16, 17])
        p_leisure = (subset[leisure_cond]['Rented Bike Count'] >= 1000).mean()
        
        return p_commute, p_leisure


    pw_commute, pw_leisure = get_demand_probs(regular_weekday)
    pwe_commute, pwe_leisure = get_demand_probs(regular_weekend)
    ph_commute, ph_leisure = get_demand_probs(weekday_holiday)


    print(f"====== '휴일'의 정체성 분석 (평일 공휴일 vs 일반 주말) ======")
    print(f"[분석 1] 출퇴근 패턴 (8시, 18시 고수요 발생 확률)")
    print(f" - 일반 평일 (기준) : {pw_commute*100:.1f}%")
    print(f" - 일반 주말        : {pwe_commute*100:.1f}%")
    print(f" - 평일 공휴일      : {ph_commute*100:.1f}%")
    
    print(f"\n[분석 2] 오후 레저 패턴 (14시~17시 고수요 발생 확률)")
    print(f" - 일반 평일 (기준) : {pw_leisure*100:.1f}%")
    print(f" - 일반 주말        : {pwe_leisure*100:.1f}%")
    print(f" - 평일 공휴일      : {ph_leisure*100:.1f}%")
    print(f"=============================================================")
    
    print(f"* 참고: 평일 공휴일 데이터는 총 {len(weekday_holiday)//24}일치 포함됨")


analyze_holiday_identity(df)

====== '휴일'의 정체성 분석 (평일 공휴일 vs 일반 주말) ======
[분석 1] 출퇴근 패턴 (8시, 18시 고수요 발생 확률)
 - 일반 평일 (기준) : 55.1%
 - 일반 주말        : 17.8%
 - 평일 공휴일      : 14.3%

[분석 2] 오후 레저 패턴 (14시~17시 고수요 발생 확률)
 - 일반 평일 (기준) : 41.9%
 - 일반 주말        : 48.6%
 - 평일 공휴일      : 37.5%
* 참고: 평일 공휴일 데이터는 총 14일치 포함됨
